# Tutorial Multi-GNN: Detección de Lavado de Dinero con Graph Neural Networks

Este notebook te guiará paso a paso por el código del repositorio **IBM Multi-GNN** para que puedas entender y ejecutar los modelos de detección de lavado de dinero (AML - Anti-Money Laundering).

## ¿Qué aprenderás?

1. **Conceptos básicos**: Qué son las GNNs y por qué son útiles para transacciones financieras
2. **Carga de datos**: Cómo se procesan las transacciones financieras como grafos
3. **Modelos**: 4 arquitecturas diferentes (GINe, GATe, PNA, RGCN)
4. **Entrenamiento**: Cómo se entrenan estos modelos paso a paso
5. **Evaluación**: Métricas y cómo interpretar resultados

## Estructura del notebook

- **Parte 1**: Setup y conceptos básicos
- **Parte 2**: Exploración de datos
- **Parte 3**: Comprensión de los modelos
- **Parte 4**: Entrenamiento completo
- **Parte 5**: Inferencia y predicción

---
# Parte 1: Setup y Conceptos Básicos

## 1.1 ¿Qué son las Graph Neural Networks (GNNs)?

Las **GNNs** son redes neuronales diseñadas para trabajar con datos en forma de grafo:
- **Nodos**: Cuentas bancarias
- **Aristas**: Transacciones entre cuentas
- **Características**: Cantidad, moneda, timestamp, etc.

### ¿Por qué GNNs para detección de lavado de dinero?

El lavado de dinero no es solo una transacción sospechosa aislada, sino **patrones de comportamiento en la red**:
- Múltiples transacciones pequeñas (smurfing)
- Cadenas de transferencias para ofuscar origen
- Patrones temporales anómalos

Las GNNs pueden capturar estos patrones porque:
1. **Message Passing**: Los nodos intercambian información con sus vecinos
2. **Agregación**: Cada nodo aprende del contexto de su vecindario
3. **Edge Prediction**: Predicen si una transacción (arista) es ilícita

## 1.2 Instalación de dependencias

In [ ]:
# Primero, vamos a verificar si tenemos las librerías necesarias
import sys
import subprocess

def check_and_install():
    """
    Verifica e instala las dependencias necesarias.
    """
    print("Verificando dependencias...\n")
    
    # Lista de paquetes a verificar
    packages = [
        'torch',
        'torch_geometric',
        'numpy',
        'pandas',
        'matplotlib',
        'scikit-learn',
        'tqdm'
    ]
    
    missing = []
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
            print(f"✓ {package} instalado")
        except ImportError:
            print(f"✗ {package} NO instalado")
            missing.append(package)
    
    if missing:
        print(f"\n⚠️ Faltan paquetes: {', '.join(missing)}")
        print("\nPara instalarlos, ejecuta en tu terminal:")
        print("conda env create -f Multi-GNN/env.yml")
        print("conda activate multignn")
    else:
        print("\n✓ Todas las dependencias están instaladas!")

check_and_install()

## 1.3 Importar librerías

In [ ]:
# Librerías estándar
import os
import sys
import json
from pathlib import Path

# Añadir el directorio Multi-GNN al path para importar sus módulos
sys.path.insert(0, os.path.join(os.getcwd(), 'Multi-GNN'))

# Ciencia de datos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

# PyTorch Geometric
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

# Scikit-learn para métricas
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# Configurar estilo de visualización
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
# Parte 2: Exploración de Datos

## 2.1 Descargar los datos

Los datos provienen de Kaggle: **IBM Transactions for Anti-Money Laundering (AML)**

**Link**: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml

### Pasos para obtener los datos:

1. Descarga el dataset de Kaggle (necesitas cuenta)
2. Descomprime el archivo
3. Ejecuta el script de formateo (si existe) o usa los datos directamente

Para este tutorial, vamos a simular datos si no están disponibles, o cargar los reales si los tienes.

In [ ]:
# Rutas de datos
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Ruta al archivo de transacciones
TRANSACTIONS_FILE = DATA_DIR / "formatted_transactions.csv"

# Verificar si existen los datos
if TRANSACTIONS_FILE.exists():
    print(f"✓ Datos encontrados en: {TRANSACTIONS_FILE}")
else:
    print(f"✗ No se encontraron datos en: {TRANSACTIONS_FILE}")
    print("\n📝 INSTRUCCIONES:")
    print("1. Descarga los datos de Kaggle:")
    print("   https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
    print("2. Descomprime y coloca el archivo CSV en la carpeta 'data/'")
    print("3. O ejecuta: python Multi-GNN/format_kaggle_files.py <ruta_al_csv>")
    print("\n⚠️ Por ahora, crearemos datos sintéticos para demostración...")

## 2.2 Crear datos sintéticos (para demostración)

Si no tienes los datos reales, vamos a crear un dataset sintético pequeño que simule transacciones financieras.

In [ ]:
def create_synthetic_data(n_transactions=1000, n_accounts=100, illicit_ratio=0.02):
    """
    Crea un dataset sintético de transacciones.
    
    Args:
        n_transactions: Número de transacciones
        n_accounts: Número de cuentas únicas
        illicit_ratio: Proporción de transacciones ilícitas
    
    Returns:
        DataFrame con transacciones sintéticas
    """
    np.random.seed(42)
    
    # Generar transacciones
    data = {
        'Timestamp': np.sort(np.random.randint(0, 86400*30, n_transactions)),  # 30 días
        'From Bank': np.random.randint(0, n_accounts, n_transactions),
        'From Account': np.random.randint(0, n_accounts, n_transactions),
        'To Bank': np.random.randint(0, n_accounts, n_transactions),
        'To Account': np.random.randint(0, n_accounts, n_transactions),
        'Amount Received': np.random.exponential(1000, n_transactions),
        'Receiving Currency': np.random.choice(['USD', 'EUR', 'GBP'], n_transactions),
        'Amount Paid': np.random.exponential(1000, n_transactions),
        'Payment Currency': np.random.choice(['USD', 'EUR', 'GBP'], n_transactions),
        'Payment Format': np.random.choice(['Reinvestment', 'Wire', 'ACH'], n_transactions),
    }
    
    # Generar etiquetas (desbalanceadas)
    n_illicit = int(n_transactions * illicit_ratio)
    labels = np.array([1] * n_illicit + [0] * (n_transactions - n_illicit))
    np.random.shuffle(labels)
    data['Is Laundering'] = labels
    
    df = pd.DataFrame(data)
    
    # Simular patrones de lavado: transacciones ilícitas tienden a ser más pequeñas y frecuentes
    illicit_mask = df['Is Laundering'] == 1
    df.loc[illicit_mask, 'Amount Received'] *= 0.3  # Cantidades menores
    
    return df

# Crear o cargar datos
if not TRANSACTIONS_FILE.exists():
    print("Creando datos sintéticos...")
    df_transactions = create_synthetic_data(n_transactions=5000, n_accounts=200)
    df_transactions.to_csv(TRANSACTIONS_FILE, index=False)
    print(f"✓ Datos sintéticos creados y guardados en: {TRANSACTIONS_FILE}")
else:
    print("Cargando datos reales...")
    df_transactions = pd.read_csv(TRANSACTIONS_FILE)
    print(f"✓ Datos cargados: {len(df_transactions)} transacciones")

# Mostrar primeras filas
print("\n📊 Primeras 5 transacciones:")
df_transactions.head()

## 2.3 Análisis exploratorio

In [ ]:
# Estadísticas básicas
print("📈 ESTADÍSTICAS DEL DATASET")
print("=" * 50)
print(f"Total transacciones: {len(df_transactions):,}")
print(f"Transacciones ilícitas: {df_transactions['Is Laundering'].sum():,}")
print(f"Transacciones lícitas: {(df_transactions['Is Laundering'] == 0).sum():,}")
print(f"Ratio ilícitas: {df_transactions['Is Laundering'].mean()*100:.2f}%")
print(f"\nCuentas únicas: {len(set(df_transactions['From Account'].unique()) | set(df_transactions['To Account'].unique()))}")
print(f"\nPeriodo temporal: {df_transactions['Timestamp'].min()} - {df_transactions['Timestamp'].max()}")
print(f"Días: {(df_transactions['Timestamp'].max() - df_transactions['Timestamp'].min()) / 86400:.1f}")

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribución de etiquetas
df_transactions['Is Laundering'].value_counts().plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Distribución de Clases (0=Lícita, 1=Ilícita)')
axes[0,0].set_ylabel('Número de transacciones')
axes[0,0].set_xlabel('Clase')

# 2. Distribución de cantidades
df_transactions.groupby('Is Laundering')['Amount Received'].hist(alpha=0.6, bins=50, ax=axes[0,1], legend=True)
axes[0,1].set_title('Distribución de Cantidades por Clase')
axes[0,1].set_xlabel('Cantidad')
axes[0,1].set_ylabel('Frecuencia')
axes[0,1].legend(['Lícita', 'Ilícita'])

# 3. Transacciones por moneda
if 'Receiving Currency' in df_transactions.columns:
    df_transactions['Receiving Currency'].value_counts().plot(kind='bar', ax=axes[1,0])
    axes[1,0].set_title('Transacciones por Moneda')
    axes[1,0].set_ylabel('Número de transacciones')

# 4. Transacciones por formato de pago
if 'Payment Format' in df_transactions.columns:
    df_transactions['Payment Format'].value_counts().plot(kind='bar', ax=axes[1,1])
    axes[1,1].set_title('Transacciones por Formato de Pago')
    axes[1,1].set_ylabel('Número de transacciones')

plt.tight_layout()
plt.show()

print("\n⚠️ NOTA IMPORTANTE: El dataset está muy DESBALANCEADO")
print("Esto es realista: en la vida real, la mayoría de transacciones son legítimas.")
print("Por eso usaremos pesos en la función de pérdida para compensar.")

## 2.4 Convertir datos a formato de grafo

Ahora vamos a transformar las transacciones en un **grafo**:
- **Nodos**: Cuentas bancarias
- **Aristas**: Transacciones (dirigidas, de cuenta origen a cuenta destino)
- **Características de aristas**: Timestamp, Amount, Currency, Payment Format
- **Etiquetas**: Is Laundering (0 o 1)

In [ ]:
def transactions_to_graph(df):
    """
    Convierte DataFrame de transacciones a objeto PyG Data.
    
    Args:
        df: DataFrame con columnas ['From Account', 'To Account', 'Timestamp', 
                                     'Amount Received', 'Receiving Currency', 
                                     'Payment Format', 'Is Laundering']
    
    Returns:
        data: Objeto torch_geometric.data.Data
    """
    # 1. Crear mapeo de IDs de cuentas a índices contiguos
    all_accounts = set(df['From Account'].unique()) | set(df['To Account'].unique())
    account_to_idx = {account: idx for idx, account in enumerate(sorted(all_accounts))}
    
    print(f"Total de cuentas (nodos): {len(account_to_idx)}")
    
    # 2. Crear edge_index (aristas del grafo)
    from_nodes = df['From Account'].map(account_to_idx).values
    to_nodes = df['To Account'].map(account_to_idx).values
    edge_index = torch.tensor([from_nodes, to_nodes], dtype=torch.long)
    
    print(f"Total de transacciones (aristas): {edge_index.shape[1]}")
    
    # 3. Crear características de nodos (simplificado: solo embeddings placeholder)
    # En el código original, los nodos no tienen features iniciales, se aprenden
    x = torch.ones(len(account_to_idx), 1)
    
    # 4. Crear características de aristas
    edge_features = []
    
    # Timestamp (normalizado)
    timestamps = df['Timestamp'].values
    timestamps_norm = (timestamps - timestamps.mean()) / (timestamps.std() + 1e-8)
    edge_features.append(timestamps_norm.reshape(-1, 1))
    
    # Amount (normalizado)
    amounts = df['Amount Received'].values
    amounts_norm = (amounts - amounts.mean()) / (amounts.std() + 1e-8)
    edge_features.append(amounts_norm.reshape(-1, 1))
    
    # Currency (one-hot encoding)
    if 'Receiving Currency' in df.columns:
        currency_dummies = pd.get_dummies(df['Receiving Currency']).values
        edge_features.append(currency_dummies)
    
    # Payment Format (one-hot encoding)
    if 'Payment Format' in df.columns:
        format_dummies = pd.get_dummies(df['Payment Format']).values
        edge_features.append(format_dummies)
    
    edge_attr = torch.tensor(np.concatenate(edge_features, axis=1), dtype=torch.float)
    
    print(f"Dimensión de características de aristas: {edge_attr.shape[1]}")
    
    # 5. Etiquetas (para cada arista)
    y = torch.tensor(df['Is Laundering'].values, dtype=torch.long)
    
    # 6. Crear objeto Data de PyTorch Geometric
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        num_nodes=len(account_to_idx)
    )
    
    # Guardar timestamps originales para análisis temporal
    data.timestamps = torch.tensor(timestamps, dtype=torch.float)
    
    return data, account_to_idx

# Convertir datos
print("Convirtiendo transacciones a grafo...\n")
graph_data, account_mapping = transactions_to_graph(df_transactions)

print("\n✓ Grafo creado exitosamente!")
print(f"\nResumen del grafo:")
print(f"  - Nodos: {graph_data.num_nodes}")
print(f"  - Aristas: {graph_data.edge_index.shape[1]}")
print(f"  - Features por nodo: {graph_data.x.shape[1]}")
print(f"  - Features por arista: {graph_data.edge_attr.shape[1]}")
print(f"  - Etiquetas positivas: {graph_data.y.sum().item()} ({graph_data.y.float().mean()*100:.2f}%)")

## 2.5 División temporal de datos

En el código original, los datos se dividen **temporalmente** por días:
- **Train**: Primeros 60% de días
- **Validation**: Siguientes 20% de días
- **Test**: Últimos 20% de días

Esto simula un escenario realista: entrenar con datos históricos y predecir en datos futuros.

In [ ]:
def temporal_split(data, train_ratio=0.6, val_ratio=0.2):
    """
    Divide el grafo temporalmente basándose en timestamps.
    
    Args:
        data: Objeto PyG Data
        train_ratio: Proporción para entrenamiento
        val_ratio: Proporción para validación
    
    Returns:
        train_mask, val_mask, test_mask: Máscaras booleanas para cada split
    """
    timestamps = data.timestamps.numpy()
    sorted_indices = np.argsort(timestamps)
    
    n_edges = len(timestamps)
    n_train = int(n_edges * train_ratio)
    n_val = int(n_edges * val_ratio)
    
    train_mask = torch.zeros(n_edges, dtype=torch.bool)
    val_mask = torch.zeros(n_edges, dtype=torch.bool)
    test_mask = torch.zeros(n_edges, dtype=torch.bool)
    
    train_mask[sorted_indices[:n_train]] = True
    val_mask[sorted_indices[n_train:n_train+n_val]] = True
    test_mask[sorted_indices[n_train+n_val:]] = True
    
    return train_mask, val_mask, test_mask

# Crear máscaras de división
train_mask, val_mask, test_mask = temporal_split(graph_data)

graph_data.train_mask = train_mask
graph_data.val_mask = val_mask
graph_data.test_mask = test_mask

print("División temporal del dataset:")
print(f"  Train: {train_mask.sum()} transacciones ({train_mask.float().mean()*100:.1f}%)")
print(f"  Val:   {val_mask.sum()} transacciones ({val_mask.float().mean()*100:.1f}%)")
print(f"  Test:  {test_mask.sum()} transacciones ({test_mask.float().mean()*100:.1f}%)")

# Verificar distribución de clases en cada split
print("\nDistribución de clases ilícitas:")
print(f"  Train: {graph_data.y[train_mask].float().mean()*100:.2f}%")
print(f"  Val:   {graph_data.y[val_mask].float().mean()*100:.2f}%")
print(f"  Test:  {graph_data.y[test_mask].float().mean()*100:.2f}%")

---
# Parte 3: Comprensión de los Modelos

## 3.1 Arquitectura general

Todos los modelos Multi-GNN siguen esta estructura:

```
INPUT (Grafo)
    ↓
EMBEDDINGS (Transforman features a espacio latente)
    ↓
GNN LAYERS (Message Passing + Agregación)
    ├─ BatchNorm (normalización)
    ├─ ReLU (activación)
    └─ Residual Connection (skip connection)
    ↓
MLP FINAL (Clasificación de aristas)
    ├─ Linear(n_hidden*3, 50)
    ├─ BatchNorm + ReLU
    ├─ Linear(50, 25)
    ├─ BatchNorm + ReLU
    └─ Linear(25, 2)  → [P(lícita), P(ilícita)]
```

### ¿Por qué n_hidden*3 en el MLP?

Para clasificar una arista (i → j), concatenamos:
1. Embedding del nodo origen (i)
2. Embedding del nodo destino (j)
3. Features de la arista

Total: 3 × n_hidden dimensiones

## 3.2 Los 4 modelos explicados

### Modelo 1: GINe (Graph Isomorphism Network with Edge features)

**Idea principal**: Basado en el algoritmo Weisfeiler-Lehman para testing de isomorfismo de grafos.

**Fórmula de actualización**:
```
h_i^(k+1) = MLP( (1 + ε) · h_i^(k) + Σ_{j∈N(i)} MLP(h_j^(k) + e_ij) )
```

Donde:
- `h_i`: Embedding del nodo i
- `e_ij`: Features de la arista i→j
- `ε`: Parámetro aprendible
- `MLP`: Red neuronal pequeña

**Ventajas**:
- Teóricamente muy expresivo
- Rápido y eficiente
- Buen baseline

**Parámetros clave**:
- `n_hidden=66`: Dimensión de embeddings
- `n_gnn_layers=2`: Número de capas GNN

### Modelo 2: GATe (Graph Attention Network with Edge features)

**Idea principal**: Usa **atención multi-cabeza** para ponderar la importancia de cada vecino.

**Fórmula de atención**:
```
α_ij = softmax_j( LeakyReLU( a^T [Wh_i || Wh_j || e_ij] ) )
h_i^(k+1) = Σ_{j∈N(i)} α_ij · W · h_j^(k)
```

Donde:
- `α_ij`: Peso de atención (¿cuán importante es el vecino j para i?)
- `||`: Concatenación
- `W`: Matriz de pesos aprendible

**Ventajas**:
- Aprende qué vecinos son más relevantes
- Multi-cabeza captura diferentes aspectos
- Interpretable (podemos visualizar atención)

**Parámetros clave**:
- `n_heads=4`: Número de cabezas de atención
- `n_hidden=64`: Dimensión por cabeza

### Modelo 3: PNA (Principal Neighbourhood Aggregation)

**Idea principal**: En lugar de una sola función de agregación, usa **múltiples agregadores**.

**Agregadores**:
- `mean`: Promedio de vecinos
- `max`: Máximo valor
- `min`: Mínimo valor
- `std`: Desviación estándar

**Escaladores** (para normalizar por grado del nodo):
- `identity`: Sin escalar
- `amplification`: Amplifica nodos con muchos vecinos
- `attenuation`: Atenúa nodos con muchos vecinos

**Ventajas**:
- Más expresivo que GIN o GAT
- Captura diferentes aspectos del vecindario
- Estado del arte en varios benchmarks

**Parámetros clave**:
- `n_hidden=20`: Dimensión (menor porque concatena múltiples agregadores)

### Modelo 4: RGCN (Relational Graph Convolutional Network)

**Idea principal**: Diseñado para **grafos multi-relacionales** (con diferentes tipos de aristas).

**Fórmula**:
```
h_i^(k+1) = σ( Σ_{r∈R} Σ_{j∈N_r(i)} W_r · h_j^(k) / |N_r(i)| )
```

Donde:
- `r`: Tipo de relación
- `W_r`: Pesos específicos para la relación r
- `N_r(i)`: Vecinos de i conectados por relación r

**En Multi-GNN**: Se usa con `--reverse_mp` para crear 2 tipos de aristas:
1. `node → node`: Transacción normal
2. `node ←rev→ node`: Transacción inversa (para capturar flujo de dinero en ambas direcciones)

**Ventajas**:
- Maneja múltiples tipos de relaciones
- Útil para grafos heterogéneos

**Parámetros clave**:
- `num_relations`: Número de tipos de aristas
- `n_hidden=66`: Dimensión

## 3.3 Implementación simplificada de GINe

Vamos a implementar una versión simplificada de GINe para entender cómo funciona.

In [ ]:
from torch_geometric.nn import GINEConv, global_add_pool

class SimpleGINe(nn.Module):
    """
    Versión simplificada de GINe para propósitos educativos.
    """
    def __init__(self, n_features, n_hidden, n_edge_features, n_layers=2):
        super().__init__()
        
        print("\n🏗️ Construyendo modelo GINe...")
        print(f"  - Input features: {n_features}")
        print(f"  - Hidden dimension: {n_hidden}")
        print(f"  - Edge features: {n_edge_features}")
        print(f"  - GNN layers: {n_layers}")
        
        # Embedding inicial de nodos
        self.node_emb = nn.Linear(n_features, n_hidden)
        
        # Embedding de características de aristas
        self.edge_emb = nn.Linear(n_edge_features, n_hidden)
        
        # Capas GNN
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        for i in range(n_layers):
            # MLP para GINE
            mlp = nn.Sequential(
                nn.Linear(n_hidden, n_hidden),
                nn.BatchNorm1d(n_hidden),
                nn.ReLU(),
                nn.Linear(n_hidden, n_hidden)
            )
            self.convs.append(GINEConv(mlp, train_eps=True))
            self.batch_norms.append(nn.BatchNorm1d(n_hidden))
        
        # MLP final para clasificación de aristas
        self.edge_classifier = nn.Sequential(
            nn.Linear(n_hidden * 3, 50),  # 3x porque concatenamos: src, dst, edge
            nn.BatchNorm1d(50),
            nn.ReLU(),
            nn.Linear(50, 25),
            nn.BatchNorm1d(25),
            nn.ReLU(),
            nn.Linear(25, 2)  # 2 clases: lícita vs ilícita
        )
        
        print("✓ Modelo construido")
    
    def forward(self, x, edge_index, edge_attr):
        """
        Forward pass del modelo.
        
        Args:
            x: Features de nodos [num_nodes, n_features]
            edge_index: Índices de aristas [2, num_edges]
            edge_attr: Features de aristas [num_edges, n_edge_features]
        
        Returns:
            logits: Predicciones para cada arista [num_edges, 2]
        """
        # 1. Embeddings iniciales
        x = self.node_emb(x)  # [num_nodes, n_hidden]
        edge_emb = self.edge_emb(edge_attr)  # [num_edges, n_hidden]
        
        # 2. Message passing a través de capas GNN
        for i, conv in enumerate(self.convs):
            x_prev = x  # Guardar para residual connection
            
            # Convolución GNN
            x = conv(x, edge_index, edge_emb)
            
            # Normalización y activación
            x = self.batch_norms[i](x)
            x = F.relu(x)
            
            # Residual connection (si las dimensiones coinciden)
            if x.shape == x_prev.shape:
                x = x + x_prev
        
        # 3. Clasificación de aristas
        # Para cada arista (i → j), concatenamos: h_i, h_j, edge_emb
        src, dst = edge_index[0], edge_index[1]
        edge_repr = torch.cat([
            x[src],      # Embedding del nodo origen
            x[dst],      # Embedding del nodo destino
            edge_emb     # Embedding de la arista
        ], dim=1)  # [num_edges, n_hidden * 3]
        
        # 4. Predicción final
        logits = self.edge_classifier(edge_repr)  # [num_edges, 2]
        
        return logits

# Crear instancia del modelo
model = SimpleGINe(
    n_features=graph_data.x.shape[1],
    n_hidden=64,
    n_edge_features=graph_data.edge_attr.shape[1],
    n_layers=2
)

# Contar parámetros
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Estadísticas del modelo:")
print(f"  - Total de parámetros: {n_params:,}")
print(f"  - Parámetros entrenables: {n_trainable:,}")

# Test forward pass
print("\n🧪 Probando forward pass...")
with torch.no_grad():
    output = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    print(f"  Input: {graph_data.num_nodes} nodos, {graph_data.edge_index.shape[1]} aristas")
    print(f"  Output shape: {output.shape} (una predicción por arista)")
    print(f"  Output example (primeras 3 aristas):")
    print(f"    {output[:3]}")
    print(f"\n  Interpretación: [logit_clase_0, logit_clase_1]")
    print(f"  Clase predicha (primeras 3): {output[:3].argmax(dim=1)}")

---
# Parte 4: Entrenamiento Completo

## 4.1 Configuración del entrenamiento

In [ ]:
# Configuración
config = {
    'lr': 0.006,              # Learning rate
    'n_epochs': 50,           # Número de épocas
    'batch_size': 128,        # Tamaño de batch (para LinkNeighborLoader)
    'n_hidden': 64,           # Dimensión oculta
    'n_gnn_layers': 2,        # Capas GNN
    'num_neighbors': [50, 50], # Vecinos a samplear por capa
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Pesos para Cross Entropy (por desbalance de clases)
    'w_ce1': 1.0,             # Peso para clase 0 (lícita)
    'w_ce2': 6.0,             # Peso para clase 1 (ilícita) - mayor para compensar
}

print("⚙️ CONFIGURACIÓN DE ENTRENAMIENTO")
print("=" * 50)
for key, value in config.items():
    print(f"  {key:20s}: {value}")

# Mover datos a device
device = torch.device(config['device'])
graph_data = graph_data.to(device)
model = model.to(device)

print(f"\n✓ Datos y modelo movidos a: {device}")

## 4.2 Función de pérdida y optimizador

In [ ]:
# Pesos para compensar desbalance de clases
class_weights = torch.tensor([config['w_ce1'], config['w_ce2']]).to(device)

# Cross Entropy Loss con pesos
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizador Adam
optimizer = Adam(model.parameters(), lr=config['lr'])

print("✓ Loss function: CrossEntropyLoss con pesos")
print(f"  - Peso clase 0 (lícita): {config['w_ce1']}")
print(f"  - Peso clase 1 (ilícita): {config['w_ce2']}")
print(f"\n✓ Optimizador: Adam con lr={config['lr']}")

## 4.3 Funciones de evaluación

In [ ]:
def evaluate(model, data, mask):
    """
    Evalúa el modelo en un subset de datos.
    
    Args:
        model: Modelo a evaluar
        data: Objeto Data con el grafo completo
        mask: Máscara booleana para seleccionar aristas
    
    Returns:
        metrics: Dict con métricas (loss, f1, precision, recall)
    """
    model.eval()
    
    with torch.no_grad():
        # Forward pass
        logits = model(data.x, data.edge_index, data.edge_attr)
        
        # Filtrar por máscara
        logits_masked = logits[mask]
        labels_masked = data.y[mask]
        
        # Loss
        loss = criterion(logits_masked, labels_masked)
        
        # Predicciones
        preds = logits_masked.argmax(dim=1)
        
        # Métricas (convertir a CPU y numpy para sklearn)
        preds_cpu = preds.cpu().numpy()
        labels_cpu = labels_masked.cpu().numpy()
        
        f1 = f1_score(labels_cpu, preds_cpu, average='binary', zero_division=0)
        precision = precision_score(labels_cpu, preds_cpu, average='binary', zero_division=0)
        recall = recall_score(labels_cpu, preds_cpu, average='binary', zero_division=0)
        
        # Accuracy
        accuracy = (preds == labels_masked).float().mean().item()
    
    return {
        'loss': loss.item(),
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'accuracy': accuracy
    }

print("✓ Función de evaluación definida")
print("  Métricas: Loss, F1-score, Precision, Recall, Accuracy")

## 4.4 Loop de entrenamiento

In [ ]:
def train_epoch(model, data, train_mask, optimizer):
    """
    Entrena el modelo por una época.
    """
    model.train()
    
    # Forward pass
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, data.edge_attr)
    
    # Calcular loss solo en training set
    loss = criterion(logits[train_mask], data.y[train_mask])
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Calcular F1 en train
    with torch.no_grad():
        preds = logits[train_mask].argmax(dim=1)
        labels = data.y[train_mask]
        f1 = f1_score(labels.cpu().numpy(), preds.cpu().numpy(), average='binary', zero_division=0)
    
    return loss.item(), f1

# Historial de entrenamiento
history = {
    'train_loss': [],
    'train_f1': [],
    'val_loss': [],
    'val_f1': [],
    'val_precision': [],
    'val_recall': [],
}

best_val_f1 = 0.0
best_epoch = 0

print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 70)

for epoch in range(config['n_epochs']):
    # Entrenar
    train_loss, train_f1 = train_epoch(model, graph_data, train_mask, optimizer)
    
    # Evaluar en validación
    val_metrics = evaluate(model, graph_data, val_mask)
    
    # Guardar historial
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_metrics['loss'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    
    # Guardar mejor modelo
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_epoch = epoch
        # En producción, aquí guardarías el modelo: torch.save(model.state_dict(), 'best_model.pt')
    
    # Imprimir progreso cada 5 épocas
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{config['n_epochs']} | "
              f"Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | "
              f"Val Loss: {val_metrics['loss']:.4f} F1: {val_metrics['f1']:.4f} "
              f"P: {val_metrics['precision']:.4f} R: {val_metrics['recall']:.4f}")

print("\n✓ ENTRENAMIENTO COMPLETADO")
print(f"  Mejor época: {best_epoch+1} con Val F1: {best_val_f1:.4f}")

## 4.5 Visualización del entrenamiento

In [ ]:
# Visualizar curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch+1})')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curva de Pérdida')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 Score
axes[1].plot(history['train_f1'], label='Train F1', linewidth=2)
axes[1].plot(history['val_f1'], label='Val F1', linewidth=2)
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch+1})')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score durante Entrenamiento')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualizar Precision vs Recall
plt.figure(figsize=(10, 5))
plt.plot(history['val_precision'], label='Precision', linewidth=2)
plt.plot(history['val_recall'], label='Recall', linewidth=2)
plt.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch+1})')
plt.xlabel('Época')
plt.ylabel('Score')
plt.title('Precision y Recall en Validación')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n📊 Interpretación:")
print("  - Loss: Debe decrecer. Si aumenta en val, hay overfitting.")
print("  - F1: Balance entre precision y recall. Mejor métrica para datos desbalanceados.")
print("  - Precision: De las predichas ilícitas, ¿cuántas son realmente ilícitas?")
print("  - Recall: De las realmente ilícitas, ¿cuántas detectamos?")

---
# Parte 5: Evaluación Final y Análisis

## 5.1 Evaluación en Test Set

In [ ]:
# Evaluar en test set
test_metrics = evaluate(model, graph_data, test_mask)

print("🎯 RESULTADOS FINALES EN TEST SET")
print("=" * 50)
print(f"  Loss:      {test_metrics['loss']:.4f}")
print(f"  F1 Score:  {test_metrics['f1']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")

# Matriz de confusión
model.eval()
with torch.no_grad():
    logits = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    preds = logits[test_mask].argmax(dim=1).cpu().numpy()
    labels = graph_data.y[test_mask].cpu().numpy()

cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Lícita', 'Ilícita'],
            yticklabels=['Lícita', 'Ilícita'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción')
plt.title('Matriz de Confusión - Test Set')
plt.show()

print("\n📊 Interpretación de la Matriz de Confusión:")
print(f"  True Negatives (TN):  {cm[0,0]:5d} - Lícitas correctamente clasificadas")
print(f"  False Positives (FP): {cm[0,1]:5d} - Lícitas clasificadas como ilícitas (falsa alarma)")
print(f"  False Negatives (FN): {cm[1,0]:5d} - Ilícitas clasificadas como lícitas (¡peligroso!)")
print(f"  True Positives (TP):  {cm[1,1]:5d} - Ilícitas correctamente detectadas")

## 5.2 Análisis de predicciones

In [ ]:
# Obtener probabilidades
model.eval()
with torch.no_grad():
    logits = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    probs = F.softmax(logits, dim=1)
    
    # Probabilidad de ser ilícita
    prob_illicit = probs[:, 1].cpu().numpy()

# Visualizar distribución de probabilidades
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribución por clase real
test_prob_licit = prob_illicit[test_mask.cpu().numpy() & (graph_data.y.cpu().numpy() == 0)]
test_prob_illicit = prob_illicit[test_mask.cpu().numpy() & (graph_data.y.cpu().numpy() == 1)]

axes[0].hist(test_prob_licit, bins=50, alpha=0.6, label='Lícitas (real)', color='green')
axes[0].hist(test_prob_illicit, bins=50, alpha=0.6, label='Ilícitas (real)', color='red')
axes[0].axvline(0.5, color='black', linestyle='--', label='Umbral (0.5)')
axes[0].set_xlabel('Probabilidad de ser Ilícita')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Probabilidades - Test Set')
axes[0].legend()
axes[0].set_yscale('log')

# Top transacciones más sospechosas
test_indices = torch.where(test_mask)[0].cpu().numpy()
top_k = 20
top_suspicious = np.argsort(prob_illicit[test_mask.cpu().numpy()])[-top_k:][::-1]
top_suspicious_global = test_indices[top_suspicious]

top_probs = prob_illicit[test_indices[top_suspicious]]
top_labels = graph_data.y[test_mask][top_suspicious].cpu().numpy()

colors = ['red' if label == 1 else 'orange' for label in top_labels]
axes[1].barh(range(top_k), top_probs, color=colors)
axes[1].set_xlabel('Probabilidad de ser Ilícita')
axes[1].set_ylabel('Ranking')
axes[1].set_title(f'Top {top_k} Transacciones Más Sospechosas')
axes[1].axvline(0.5, color='black', linestyle='--', alpha=0.5)
axes[1].invert_yaxis()

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Realmente Ilícita'),
    Patch(facecolor='orange', label='Realmente Lícita (Falso Positivo)')
]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print(f"\n🔍 Top {top_k} transacciones más sospechosas:")
print(f"  Correctas (TP): {(top_labels == 1).sum()} / {top_k}")
print(f"  Falsas alarmas (FP): {(top_labels == 0).sum()} / {top_k}")

## 5.3 Inferencia en nuevas transacciones

Así es como usarías el modelo entrenado para clasificar nuevas transacciones.

In [ ]:
def predict_transaction(model, graph_data, edge_index_to_predict):
    """
    Predice si una transacción específica es ilícita.
    
    Args:
        model: Modelo entrenado
        graph_data: Grafo completo
        edge_index_to_predict: Índice de la arista a predecir
    
    Returns:
        prediction: 0 (lícita) o 1 (ilícita)
        probability: Probabilidad de ser ilícita
    """
    model.eval()
    
    with torch.no_grad():
        # Forward pass
        logits = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
        
        # Obtener predicción para la arista específica
        logit = logits[edge_index_to_predict]
        prob = F.softmax(logit, dim=0)
        
        prediction = logit.argmax().item()
        probability = prob[1].item()  # Probabilidad de clase 1 (ilícita)
    
    return prediction, probability

# Ejemplo: predecir algunas transacciones del test set
print("🔮 EJEMPLOS DE PREDICCIÓN")
print("=" * 70)

# Seleccionar 5 transacciones aleatorias del test set
test_indices = torch.where(test_mask)[0]
sample_indices = test_indices[torch.randperm(len(test_indices))[:5]]

for i, idx in enumerate(sample_indices):
    # Información de la transacción
    src, dst = graph_data.edge_index[:, idx]
    real_label = graph_data.y[idx].item()
    
    # Predicción
    pred_label, pred_prob = predict_transaction(model, graph_data, idx)
    
    # Mostrar
    print(f"\nTransacción {i+1}:")
    print(f"  Cuenta origen → destino: {src.item()} → {dst.item()}")
    print(f"  Etiqueta real: {'Ilícita' if real_label == 1 else 'Lícita'}")
    print(f"  Predicción: {'Ilícita' if pred_label == 1 else 'Lícita'} (prob={pred_prob:.3f})")
    print(f"  ¿Correcto? {'✓' if pred_label == real_label else '✗'}")

print("\n" + "=" * 70)

---
# Parte 6: Usando el Código Original Multi-GNN

## 6.1 Cómo ejecutar el código original

Ahora que entiendes cómo funcionan los modelos, puedes ejecutar el código original de Multi-GNN.

### Preparación de datos:

1. Descarga los datos de Kaggle
2. Formatea el CSV (si es necesario)
3. Actualiza `Multi-GNN/data_config.json` con las rutas correctas

### Comandos de entrenamiento:

```bash
# Ir al directorio Multi-GNN
cd Multi-GNN

# Entrenar modelo GIN básico
python main.py --data Small_HI --model gin

# Entrenar GIN con todas las adaptaciones Multi-GNN
python main.py --data Small_HI --model gin --emlps --reverse_mp --ego --ports --tds

# Entrenar otros modelos
python main.py --data Small_HI --model gat
python main.py --data Small_HI --model pna
python main.py --data Small_HI --model rgcn --reverse_mp

# Guardar modelo
python main.py --data Small_HI --model gin --save_model --unique_name my_gin_model

# Inferencia con modelo guardado
python main.py --data Small_HI --model gin --inference --unique_name my_gin_model
```

### Parámetros importantes:

- `--data`: Nombre del dataset (carpeta en data_config.json)
- `--model`: gin, gat, pna, o rgcn
- `--emlps`: Activa Edge MLPs
- `--reverse_mp`: Message passing inverso (grafo heterogéneo)
- `--ego`: Añade ego IDs a nodos centrales
- `--ports`: Numeración de puertos (orden temporal de transacciones)
- `--tds`: Time deltas entre transacciones
- `--n_epochs`: Número de épocas (default: 100)
- `--batch_size`: Tamaño de batch (default: 8192)

## 6.2 Estructura de archivos esperada

```
Multi-GNN/
├── main.py              # Punto de entrada
├── models.py            # Definiciones de modelos
├── training.py          # Lógica de entrenamiento
├── data_loading.py      # Carga de datos
├── data_util.py         # Utilidades de datos
├── train_util.py        # Utilidades de entrenamiento
├── inference.py         # Inferencia
├── util.py              # Utilidades generales
├── data_config.json     # Configuración de rutas de datos
├── model_settings.json  # Hiperparámetros por modelo
└── env.yml              # Dependencias conda

data/
└── formatted_transactions.csv  # Tus datos

logs/
└── logs.log             # Logs de entrenamiento

chkpts/
└── [modelos guardados]  # Checkpoints
```

## 6.3 Personalizar hiperparámetros

Edita `Multi-GNN/model_settings.json`:

```json
{
  "gin": {
    "params": {
      "lr": 0.00621,        // Learning rate
      "n_hidden": 66,       // Dimensión oculta
      "n_gnn_layers": 2,    // Capas GNN
      "w_ce1": 1.0,         // Peso clase lícita
      "w_ce2": 6.27         // Peso clase ilícita
    }
  }
}
```

---
# Conclusiones y Próximos Pasos

## ¿Qué aprendiste?

1. **Conceptos de GNN**: Message passing, agregación, edge prediction
2. **Datos como grafos**: Transacciones → aristas, cuentas → nodos
3. **4 arquitecturas**: GINe, GATe, PNA, RGCN
4. **Entrenamiento**: Loss, optimización, evaluación
5. **Métricas**: F1, precision, recall (más importantes que accuracy)
6. **Desbalance de clases**: Uso de pesos en loss function

## Próximos pasos sugeridos:

### 1. Experimentar con datos reales
- Descarga el dataset de Kaggle
- Ejecuta el código original con diferentes modelos
- Compara resultados

### 2. Probar adaptaciones Multi-GNN
- `--emlps`: Mejora actualización de aristas
- `--ports`: Captura orden temporal
- `--tds`: Captura tiempo entre transacciones
- `--reverse_mp`: Message passing bidireccional

### 3. Optimización de hiperparámetros
- Ajusta learning rate
- Prueba diferentes dimensiones ocultas
- Varía el número de capas GNN
- Experimenta con pesos de clase

### 4. Análisis avanzado
- Visualiza embeddings con t-SNE/UMAP
- Analiza errores (FP y FN)
- Estudia patrones de transacciones ilícitas detectadas
- Interpreta pesos de atención (en GAT)

### 5. Extensiones
- Añade más features (geolocalización, tipo de negocio, etc.)
- Implementa técnicas de explicabilidad (GNNExplainer)
- Prueba otras arquitecturas (GraphSAINT, GraphSAGE)
- Deploy en producción con API REST

## Recursos adicionales:

- **Paper Multi-GNN**: [Buscar en Google Scholar]
- **PyTorch Geometric**: https://pytorch-geometric.readthedocs.io/
- **GNN explicado**: http://snap.stanford.edu/class/cs224w-2021/
- **Kaggle dataset**: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml

## Preguntas frecuentes:

**P: ¿Por qué F1 y no accuracy?**
R: Porque los datos están muy desbalanceados. Un modelo que prediga todo como "lícita" tendría 99%+ accuracy pero sería inútil. F1 balancea precision y recall.

**P: ¿Cuántas épocas debo entrenar?**
R: Depende. Usa early stopping: para cuando val_loss deje de mejorar por N épocas (e.g., 10).

**P: ¿GPU es necesaria?**
R: No es estrictamente necesaria para datasets pequeños, pero acelera mucho el entrenamiento (10-100x).

**P: ¿Qué modelo usar?**
R: Empieza con GIN (más simple). Si no funciona bien, prueba GAT o PNA. RGCN solo si tienes múltiples tipos de relaciones.

**P: ¿Cómo interpretar las predicciones?**
R: La probabilidad indica confianza. Un threshold de 0.5 es estándar, pero puedes ajustarlo según el trade-off precision/recall que necesites.

---

## ¡Éxito con tus experimentos! 🚀

Si tienes dudas, consulta la documentación de PyTorch Geometric o el código original en `Multi-GNN/`.